In [11]:
# Cell 1 — Import libraries
import pandas as pd
import numpy as np

print("Setup complete!")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

Setup complete!
pandas: 3.0.2
numpy: 2.4.4


In [12]:
# Cell 2 — Seed and size
np.random.seed(42)
n = 400

print(f"Will generate {n} rows of product launch data")

Will generate 400 rows of product launch data


In [13]:
# Cell 3 — Generate feature columns
df = pd.DataFrame({
    'price': np.random.randint(500, 50001, n),

    'marketing_budget': np.random.randint(1000, 500001, n),

    'competition': np.random.choice(
        ['Low', 'Medium', 'High'], n
    ),

    'category': np.random.choice(
        ['Electronics', 'Clothing', 'Food', 'Health'], n
    ),

    'timing': np.random.choice(
        ['Festival', 'Normal'], n
    ),

    'region': np.random.choice(
        ['Urban', 'Rural'], n
    )
})
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

Shape: (400, 6)

First 5 rows:
   price  marketing_budget competition     category    timing region
0  16295            140788        High       Health  Festival  Urban
1   1360            220963         Low         Food    Normal  Rural
2  38658            481761         Low  Electronics    Normal  Urban
3  45232            249710      Medium     Clothing  Festival  Rural
4  11784            221984      Medium  Electronics  Festival  Rural


In [14]:
# Cell 4 — Sanity check
print("=== Numerical columns ===")
print(df[['price', 'marketing_budget']].describe())

print("\n=== Categorical columns ===")
for col in ['competition', 'category', 'timing', 'region']:
    print(f"\n{col} value counts:")
    print(df[col].value_counts())

=== Numerical columns ===
              price  marketing_budget
count    400.000000        400.000000
mean   24640.915000     234414.812500
std    15055.319494     144184.936671
min      661.000000       2542.000000
25%    11169.500000     114285.750000
50%    24131.500000     220946.500000
75%    38886.750000     356647.500000
max    49877.000000     496972.000000

=== Categorical columns ===

competition value counts:
competition
Medium    141
Low       135
High      124
Name: count, dtype: int64

category value counts:
category
Food           110
Electronics    100
Clothing        99
Health          91
Name: count, dtype: int64

timing value counts:
timing
Festival    216
Normal      184
Name: count, dtype: int64

region value counts:
region
Rural    207
Urban    193
Name: count, dtype: int64


In [15]:
# Cell 6 — Apply and check balance
# Cell 5 — NEW scoring function with interaction effects
def assign_success(row):
    score = 0

    # Base factors
    if row['competition'] == 'Low':
        score += 3
    elif row['competition'] == 'Medium':
        score += 1

    if row['marketing_budget'] > 200000:
        score += 2

    if row['timing'] == 'Festival':
        score += 2

    if row['price'] < 10000:
        score += 1

    if row['region'] == 'Urban':
        score += 1

    # Interaction effects — these make it non-linear
    # LR cannot capture these, but RF and DT can
    if row['competition'] == 'Low' and row['marketing_budget'] > 250000:
        score += 2   # synergy: low competition + big budget = big win

    if row['timing'] == 'Festival' and row['region'] == 'Urban':
        score += 1   # festival launches in urban areas do better

    if row['competition'] == 'High' and row['marketing_budget'] < 50000:
        score -= 2   # tough market + low spend = punished

    if row['price'] > 30000 and row['competition'] == 'High':
        score -= 1   # expensive product in crowded market

    # Noise
    score += np.random.randint(-1, 2)

    return 1 if score >= 5 else 0

In [16]:
# Cell 6 — Apply and CHECK BALANCE carefully
df['success'] = df.apply(assign_success, axis=1)

print("Class distribution:")
print(df['success'].value_counts())
print("\nPercentage:")
print((df['success'].value_counts(normalize=True) * 100).round(1))

Class distribution:
success
0    207
1    193
Name: count, dtype: int64

Percentage:
success
0    51.7
1    48.2
Name: proportion, dtype: float64


In [17]:
# Cell 7 — Final preview
print("Final dataset shape:", df.shape)

print("\nColumn names and types:")
print(df.dtypes)

print("\nAny missing values?")
print(df.isnull().sum())

print("\nSample rows:")
print(df.head(10))

Final dataset shape: (400, 7)

Column names and types:
price               int64
marketing_budget    int64
competition           str
category              str
timing                str
region                str
success             int64
dtype: object

Any missing values?
price               0
marketing_budget    0
competition         0
category            0
timing              0
region              0
success             0
dtype: int64

Sample rows:
   price  marketing_budget competition     category    timing region  success
0  16295            140788        High       Health  Festival  Urban        1
1   1360            220963         Low         Food    Normal  Rural        1
2  38658            481761         Low  Electronics    Normal  Urban        1
3  45232            249710      Medium     Clothing  Festival  Rural        0
4  11784            221984      Medium  Electronics  Festival  Rural        1
5   6765            373771        High         Food  Festival  Rural        0
6

In [18]:
# Cell 8 — Save main dataset
df.to_csv('../data/dataset.csv', index=False)

print("dataset.csv saved to data/ folder!")
print("Rows saved:", len(df))
print("Columns saved:", list(df.columns))

dataset.csv saved to data/ folder!
Rows saved: 400
Columns saved: ['price', 'marketing_budget', 'competition', 'category', 'timing', 'region', 'success']


In [19]:
# Cell 9 — Save demo inputs
demo_df = df.sample(20, random_state=42)
demo_df.to_csv('../data/demo_inputs.csv', index=False)

print("demo_inputs.csv saved to data/ folder!")
print("Demo rows:", len(demo_df))
print("\nPreview of demo data:")
print(demo_df[['price', 'marketing_budget',
               'competition', 'timing', 'success']].to_string())


demo_inputs.csv saved to data/ folder!
Demo rows: 20

Preview of demo data:
     price  marketing_budget competition    timing  success
209  47145            158164        High    Normal        0
280  21432            238027         Low  Festival        1
33    1767            450877        High    Normal        1
210   1354             12338         Low  Festival        1
93    9071            177615      Medium  Festival        0
84   23747            379496         Low    Normal        1
329  47833            486008         Low    Normal        1
94   40476            188628         Low    Normal        0
266  20945             52934        High    Normal        0
126  43030            240833      Medium  Festival        1
9    47691             78575      Medium  Festival        0
361  31716            174416        High  Festival        0
56   15041            463894      Medium    Normal        0
72     661            104481        High  Festival        1
132  27766            45

In [20]:
# Cell 10 — Confirm files saved
import os

files = ['../data/dataset.csv', '../data/demo_inputs.csv']

for f in files:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"EXISTS: {f}  ({size} bytes)")
    else:
        print(f"MISSING: {f}  — re-run the save cell!")

EXISTS: ../data/dataset.csv  (16921 bytes)
EXISTS: ../data/demo_inputs.csv  (927 bytes)
